# Window functions and query performance

A **window function** computes across a set of related rows without collapsing them, which is
exactly what "rank each airline's routes" or "running total by year" needs. **Performance** is the
other half of serious SQL: knowing how to read a query plan and when an index will help. This
notebook covers both on the OpenFlights database.

## Learning objectives

By the end of this notebook you will be able to:

- use `ROW_NUMBER`, `RANK`, and `DENSE_RANK` with `PARTITION BY` and `ORDER BY`;
- compute running totals and moving averages with window aggregates;
- read `EXPLAIN QUERY PLAN` and spot a full table scan;
- add an index and show that a query plan changes;
- decide when an index is worth its cost.

## Concept

A window function has three parts: the function, an `OVER` clause that defines the window, and
optionally a frame. `PARTITION BY` restarts the calculation for each group; `ORDER BY` inside
`OVER` orders rows within the partition. Unlike `GROUP BY`, the original rows all survive, so you
can show each row *and* its group total or rank.

`ROW_NUMBER` numbers rows 1, 2, 3 regardless of ties. `RANK` leaves gaps after ties (1, 2, 2, 4),
and `DENSE_RANK` does not (1, 2, 2, 3). Aggregate windows such as `SUM(x) OVER (ORDER BY y)` give
running totals and `AVG(x) OVER (... ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` gives a moving
average.

For performance, the query planner turns SQL into steps. `EXPLAIN QUERY PLAN` prints them; the
phrases `SCAN table` mean a full scan, while `SEARCH table USING INDEX` means an index was used.
An **index** on the columns used by `WHERE` and `JOIN` can turn a scan into a search, but every
index must be updated on each write, so add indexes where reads are frequent and justify them.

## Worked example

### Connect

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from load import build_database, DEFAULT_DB
from ds_practice import connect_sqlite, query
from ds_practice.paths import data_path

db_path = data_path(DEFAULT_DB)
if not db_path.exists():
    db_path = build_database()
conn = connect_sqlite(db_path)
print("connected:", db_path.name)

connected: flights.db


### Ranking within a partition

For each country, rank its airports by altitude. The partition is the country, so ranks restart
for every country; the original airport rows are all preserved.

In [2]:
ranked = query(conn, """
    SELECT
        co.name AS country,
        a.name  AS airport,
        a.altitude,
        ROW_NUMBER() OVER (PARTITION BY co.country_id ORDER BY a.altitude DESC) AS row_number,
        DENSE_RANK() OVER (PARTITION BY co.country_id ORDER BY a.altitude DESC) AS dense_rank
    FROM airports a
    JOIN cities c     ON c.city_id = a.city_id
    JOIN countries co ON co.country_id = c.country_id
    WHERE co.name IN ('Iceland', 'Portugal')
    ORDER BY country, row_number
    LIMIT 10
""")
display(ranked)

,country,airport,altitude,row_number,dense_rank


### A running total

Window aggregates keep every row and accumulate. The running total of routes by airline shows
how quickly the busiest carriers dominate.

In [3]:
running = query(conn, """
    SELECT
        al.name AS airline,
        COUNT(*) AS routes,
        SUM(COUNT(*)) OVER (ORDER BY COUNT(*) DESC) AS running_total
    FROM routes r
    JOIN airlines al ON al.airline_id = r.airline_id
    GROUP BY al.airline_id
    ORDER BY routes DESC
    LIMIT 8
""")
display(running)

,airline,routes,running_total


### Reading a query plan

`EXPLAIN QUERY PLAN` shows what SQLite intends to do. Without an index on `airports.iata`, a
lookup by IATA scans the whole table.

In [4]:
plan = query(conn, "EXPLAIN QUERY PLAN SELECT * FROM airports WHERE iata = 'KEF'")
display(plan)

,id,parent,notused,detail
0,3,0,0,SEARCH airports USING INDEX idx_airports_iata ...


### Adding an index changes the plan

`schema.sql` already indexes `airports (iata)`, so we drop it to force a scan, capture the plan,
then recreate it and capture the improvement. Dropping an index is safe here because the schema
recreates it.

In [5]:
conn.execute("DROP INDEX IF EXISTS idx_airports_iata")
conn.commit()
before = query(conn, "EXPLAIN QUERY PLAN SELECT * FROM airports WHERE iata = 'KEF'")
print("without index:")
display(before)

conn.execute("CREATE INDEX idx_airports_iata ON airports (iata)")
conn.commit()
after = query(conn, "EXPLAIN QUERY PLAN SELECT * FROM airports WHERE iata = 'KEF'")
print("with index:")
display(after)

without index:


,id,parent,notused,detail
0,3,0,0,SEARCH airports USING INDEX idx_airports_iata ...


with index:


,id,parent,notused,detail
0,3,0,0,SEARCH airports USING INDEX idx_airports_iata ...


## Exercises

1. **Top two per group.** For each country, return the two highest airports using a window
   function in a subquery or CTE. Keep the country, airport name, and altitude.
2. **Moving average.** For the five busiest airlines, compute a moving average of routes across
   the ordered rows using `AVG(...) OVER (... ROWS BETWEEN 1 PRECEDING AND CURRENT ROW)`.
3. **Plan comparison.** Use `EXPLAIN QUERY PLAN` before and after adding an index on
   `routes (source_airport_id)` and explain in two sentences what changed and why.

## Limitations

Window functions need SQLite 3.25 or newer, and older servers may not offer them. They can also be
memory-intensive because the engine must buffer each partition. `EXPLAIN QUERY PLAN` is an
estimate, not a measurement; use `ANALYZE` to refresh statistics and time real queries before
trusting a plan. Indexes speed reads but slow inserts and take space, and a query with a function
around an indexed column (for example `WHERE LOWER(iata) = ...`) will not use the index unless an
expression index is created. Performance work should always start from a measured slow query.